# Exploring the "Is it toxic?" pipelines

This notebook is a playground for understanding the two classification pipelines:

| Pipeline | Features | Classifier |
|----------|----------|------------|
| 1 | `BAAI/bge-small-en-v1.5` embeddings | Logistic Regression |
| 2 | TF-IDF | LightGBM |

It has three parts:
1. **Evaluation** — how well does each pipeline do on the eval set? (metrics + confusion matrices)
2. **Explain one input** — type a comment and see each model's probabilities.
3. **Classification space** — project the BGE embeddings to 2D and *see* how toxic / non-toxic comments separate.

> Run the ingestion and classification scripts first so the models exist:
> `python ingest_transformers.py`, `python ingest_tfidf.py`,
> `python classify_logreg.py`, `python classify_lightgbm.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression

from dataset import load_dataset, TRAINING_CSV, EVAL_CSV, LABELS
import predict  # loads both trained models (takes a few seconds)

plt.rcParams["figure.figsize"] = (6, 5)
print("Models loaded. Labels:", LABELS)

## 1. Evaluation

Run every eval comment through both pipelines and compare predictions to the true labels.

In [ ]:
eval_data = load_dataset(EVAL_CSV)
texts, y_true = eval_data.texts, eval_data.labels

# Predict with each pipeline.
y_bge   = [predict.predict_bge(t).label   for t in texts]
y_tfidf = [predict.predict_tfidf(t).label for t in texts]

print("=== Pipeline 1: BGE + Logistic Regression ===")
print(classification_report(y_true, y_bge, zero_division=0))
print("=== Pipeline 2: TF-IDF + LightGBM ===")
print(classification_report(y_true, y_tfidf, zero_division=0))

In [ ]:
# Confusion matrices side by side.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, y_pred, title in [
    (axes[0], y_bge,   "BGE + LogReg"),
    (axes[1], y_tfidf, "TF-IDF + LightGBM"),
]:
    cm = confusion_matrix(y_true, y_pred, labels=list(LABELS))
    ConfusionMatrixDisplay(cm, display_labels=list(LABELS)).plot(ax=ax, colorbar=False, cmap="Blues")
    acc = np.mean(np.array(y_true) == np.array(y_pred))
    ax.set_title(f"{title}  (acc={acc:.2f})")
plt.tight_layout()
plt.show()

## 2. Explain one input

Change `sample_text` below and re-run. You'll see the probability each model
assigns to *toxic* vs *non_toxic* — a quick way to feel how confident (or torn)
each pipeline is on a given comment.

In [ ]:
sample_text = "Nobody asked for your opinion, so maybe just sit this one out."

p_bge   = predict.predict_bge(sample_text)
p_tfidf = predict.predict_tfidf(sample_text)

print(f"Comment: {sample_text!r}\n")
print(f"BGE + LogReg      -> {p_bge.label:9s} (confidence {p_bge.confidence:.2f})")
print(f"TF-IDF + LightGBM -> {p_tfidf.label:9s} (confidence {p_tfidf.confidence:.2f})")

# Grouped bar chart of the class probabilities from both models.
labels = list(LABELS)
bge_probs   = [p_bge.probabilities[l]   for l in labels]
tfidf_probs = [p_tfidf.probabilities[l] for l in labels]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - w/2, bge_probs,   w, label="BGE + LogReg")
ax.bar(x + w/2, tfidf_probs, w, label="TF-IDF + LightGBM")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("probability"); ax.set_ylim(0, 1)
ax.set_title("Predicted probabilities per class")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.legend()
plt.show()

## 3. The classification space

The BGE model turns each comment into a 384-dimensional vector. We can't see 384
dimensions, so we **project down to 2D** and plot every comment, colored by its
true label. If toxic and non-toxic comments form separate clouds, the embeddings
carry a usable signal (which is why the Logistic Regression does well).

We use two projections:
- **PCA** — a linear projection, fast and faithful to global structure.
- **t-SNE** — a non-linear projection that emphasizes local clustering.

In [ ]:
# Embed the full training + eval set with BGE.
train = load_dataset(TRAINING_CSV)
all_texts  = train.texts + eval_data.texts
all_labels = np.array(train.labels + eval_data.labels)
split = np.array(["train"] * len(train.texts) + ["eval"] * len(eval_data.texts))

X = predict.embed(all_texts)          # (n_samples, 384)
print("Embedding matrix:", X.shape)

In [ ]:
# Project to 2D two different ways.
pca_2d  = PCA(n_components=2, random_state=42).fit_transform(X)
tsne_2d = TSNE(n_components=2, perplexity=15, random_state=42, init="pca").fit_transform(X)

def scatter(ax, coords, title):
    for label, color in zip(LABELS, ["tab:red", "tab:green"]):
        m = all_labels == label
        ax.scatter(coords[m, 0], coords[m, 1], c=color, label=label, alpha=0.7, s=40, edgecolors="k", linewidths=0.3)
    ax.set_title(title); ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter(axes[0], pca_2d,  "PCA projection of BGE embeddings")
scatter(axes[1], tsne_2d, "t-SNE projection of BGE embeddings")
plt.tight_layout(); plt.show()

### Decision boundary + your sample point

To *see* a decision boundary we train a helper Logistic Regression **on the 2D PCA
coordinates** (not the real 384-D model — this is purely for intuition). Then we
drop the `sample_text` from Part 2 onto the same map as a black star, so you can
see where it lands relative to the two clouds and the boundary.

In [ ]:
# Helper 2D model just for drawing a boundary.
boundary_model = LogisticRegression(max_iter=1000).fit(pca_2d, all_labels)

# Grid over the plot area.
pad = 1.0
x_min, x_max = pca_2d[:, 0].min() - pad, pca_2d[:, 0].max() + pad
y_min, y_max = pca_2d[:, 1].min() - pad, pca_2d[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
# Probability of "toxic" across the grid.
toxic_idx = list(boundary_model.classes_).index("toxic")
zz = boundary_model.predict_proba(grid)[:, toxic_idx].reshape(xx.shape)

# Project the sample_text into the same PCA space.
sample_vec = predict.embed([sample_text])
sample_2d  = PCA(n_components=2, random_state=42).fit(X).transform(sample_vec)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, zz, levels=20, cmap="RdYlGn_r", alpha=0.35)
ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=1.5)  # the boundary
for label, color in zip(LABELS, ["tab:red", "tab:green"]):
    m = all_labels == label
    ax.scatter(pca_2d[m, 0], pca_2d[m, 1], c=color, label=label, alpha=0.8, s=40, edgecolors="k", linewidths=0.3)
ax.scatter(sample_2d[0, 0], sample_2d[0, 1], c="black", marker="*", s=400, label="sample_text", edgecolors="white", linewidths=1)
ax.set_title("2D decision boundary (PCA space) with sample point")
ax.legend()
plt.show()

print(f"sample_text: {sample_text!r}")
print("Real BGE+LogReg prediction:", predict.predict_bge(sample_text).label)

**Reading the plots:** the red/green shading is the helper model's toxic-probability,
and the black line is its 0.5 boundary. The black star is your `sample_text`. If the
star sits deep in one cloud, both the 2D helper and the real 384-D model will agree
confidently; near the boundary the models are less sure. Remember the real classifier
sees all 384 dimensions, so it can separate points that *look* overlapping here.